# 53. Capstone: a leakage-safe telemetry classifier

**Level:** Advanced

**Prerequisites:** 11-21 and 49-51: evaluation, monitoring, and temporal assumptions.

[Course index](00_START_HERE.ipynb) | [Previous](52_Agents_Workflows_and_Bounded_Tools.ipynb) | [Next](54_Capstone_Evidence_Assistant_and_Evals.ipynb)

**Topics:** Capstone, Supervised learning, Baseline, Pipeline, Held-out test, Network engineering.

**Matching script:** `examples\53_Capstone_Network_Telemetry_Classifier.py`

**How to study:** read the notes, predict an example's output, run it, trace each change in data/shape/state, then complete the practice task. Every worked code cell contains its own imports and data and can run independently of other notebooks.

**Execution contract:** Python 3.11+, course dependencies, CPU only, synthetic/local data. No hosted API requests, model-weight downloads, dataset downloads, listening web servers, credentials, or real device changes. The temporary-file lesson cleans up its own files. Expected errors are caught narrowly and asserted; unexpected errors must remain visible.

## Concept notes and mental model

An end-to-end learning system begins with a task definition and the data available at inference.
This capstone predicts a synthetic incident label from utilization, packet loss, and latency.
It uses generated independent rows so a random split is appropriate for this demonstration,
while real device telemetry usually needs grouped or chronological evaluation.

Reserve the test set, build a preprocessing-plus-model pipeline, compare with a baseline, and
inspect class-sensitive metrics. A high score on synthetic labels only confirms that the model
can learn the manufactured relationship; it is not evidence of outage prediction on live networks.

When adapting to real data, document the prediction horizon, label source, delayed labels,
missing data, grouping, costs of missed incidents versus false alarms, and a safe response
policy. A classifier's alert is not authorization to reconfigure equipment.

## Mathematics or data contract

The synthetic label depends on a chosen score; F1 summarizes positive-class precision and recall for the fixed threshold.

## Important limitation

No live telemetry, real incident labels, or remediation tools are used.

Distinguish what is implemented and asserted below from what would need additional data, training, independent evaluation, authorization, or operational controls.

## Worked example 1: Train, compare, and report a complete small model

**Purpose:** Combine earlier pieces without hidden downloads.

### Read the code in this order
1. Generate synthetic features with explicit units.
2. Reserve a stratified test set.
3. Fit scaling plus logistic regression and a dummy baseline.
4. Assert improvement and print a classification report.

### Syntax and behavior to notice
- `fit(...)` learns model or preprocessing state from the supplied training data.
- `predict(...)` returns the estimator's chosen output, not a guarantee of correctness.
- `assert` checks the stated behavior; a failed check raises AssertionError.
- A fixed random seed makes the chosen sampling reproducible within the environment.

### Expected output
This is the output captured from executing the worked example in the recorded environment. Package versions and floating-point rounding can change numeric details; the assertions express the required shapes, invariants, and thresholds.

```text
test F1: 0.941 baseline F1: 0.0
              precision    recall  f1-score   support

           0       0.99      0.93      0.96        76
           1       0.91      0.98      0.94        49

    accuracy                           0.95       125
   macro avg       0.95      0.96      0.95       125
weighted avg       0.95      0.95      0.95       125
```

**Before running:** identify the input variables, which values are learned versus chosen, and what each assertion proves. A successful shape test does not by itself prove predictive quality. Training-only loss checks are labeled as such.

In [1]:
import numpy as np
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(42)
X = np.column_stack([rng.uniform(0, 100, 500), rng.uniform(0, 5, 500), rng.uniform(1, 100, 500)])
incident_score = 0.03*X[:, 0] + 0.7*X[:, 1] + 0.02*X[:, 2]
y = (incident_score + rng.normal(0, 0.2, 500) > 4.7).astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=42)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=300)).fit(X_train, y_train)
baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
prediction = model.predict(X_test)
score = f1_score(y_test, prediction)
baseline_score = f1_score(y_test, baseline.predict(X_test), zero_division=0)
assert score > baseline_score and score > 0.8
print("test F1:", round(score, 3), "baseline F1:", round(baseline_score, 3))
print(classification_report(y_test, prediction, zero_division=0))

test F1: 0.941 baseline F1: 0.0
              precision    recall  f1-score   support

           0       0.99      0.93      0.96        76
           1       0.91      0.98      0.94        49

    accuracy                           0.95       125
   macro avg       0.95      0.96      0.95       125
weighted avg       0.95      0.95      0.95       125



### Interpret example 1

The result should demonstrate this purpose: **Combine earlier pieces without hidden downloads.**

Follow the assertions back to the exact data or state being checked. Where a model was fitted, distinguish training fit from held-out evaluation. Where input is rejected, inspect whether rejection happens before any intended mutation or tool execution. Do not delete a failed assertion to make an incorrect result look valid.

## Practice: extend the concept yourself

Create a grouped device dataset and hold out entire devices before comparing candidate models.

1. Write your expected result before editing code.
2. Add one normal-case test and one boundary/failure-case test.
3. Keep training/validation/test responsibilities separate where relevant.
4. Report an honest failure or unknown result rather than a success-shaped fallback.

The task is intentionally not filled in for you; the worked examples above are complete and runnable. Add your own code cell for the extension.

## Understanding checkpoint
- Explain `Capstone` in your own words and give one example of when it is useful.
- Trace one worked example from input through transformation to returned or printed output.
- State this lesson's mathematical/data contract: The synthetic label depends on a chosen score; F1 summarizes positive-class precision and recall for the fixed threshold.
- Name the boundary of the demonstration: No live telemetry, real incident labels, or remediation tools are used.
- Describe one test that could catch a plausible but wrong implementation.

## Primary references and further reading

- `https://scikit-learn.org/stable/user_guide.html`

Research snapshot: **2026-10-17**. Foundational references need not be recent. Mutable library docs describe their current version; compare with the tested dependency versions in the course manifest. References are reading material, not runtime downloads.